In [0]:
dbutils.widgets.text("catalog", "bank_dev")
c = dbutils.widgets.get("catalog")

CHECKS = [
    ("daily volume is not empty",
     f"SELECT COUNT(*) > 0 FROM {c}.gold.daily_branch_txn_volume"),
    ("risk table is not empty",
     f"SELECT COUNT(*) > 0 FROM {c}.gold.high_risk_customers"),
    ("monthly balance table is not empty",
     f"SELECT COUNT(*) > 0 FROM {c}.gold.monthly_account_balance"),
    ("no duplicate transaction ids",
     f"SELECT COUNT(*) = COUNT(DISTINCT transaction_id) FROM {c}.silver.transactions"),
    ("gold reconciles to silver",
     f"SELECT (SELECT SUM(txn_count) FROM {c}.gold.daily_branch_txn_volume) "
     f"= (SELECT COUNT(*) FROM {c}.silver.transactions)"),
    ("quarantine rate is under 5%",
     f"SELECT (SELECT COUNT(*) FROM {c}.silver.transactions_quarantine) "
     f"< 0.05 * (SELECT COUNT(*) FROM {c}.bronze.transactions)"),
    ("fraud rate is a valid percentage",
     f"SELECT MAX(fraud_rate_pct) BETWEEN 0 AND 100 FROM {c}.gold.daily_branch_txn_volume"),
]

failed = [name for name, sql in CHECKS if not spark.sql(sql).first()[0]]

if failed:
    raise RuntimeError(f"Post-run checks failed: {failed}")
print(f"All {len(CHECKS)} post-run checks passed for catalog {c}")